## Gemma-2-2bのロード

In [ ]:
import torch
from transformer_lens import HookedTransformer

# 1. 高速化設定
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 2. モデルをロード（ここを実行しないと model という名前が生まれません）
model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device)

## Gemma-2-2bのプロンプト

In [ ]:
# プロンプト入力
input_text = "入力テキスト"

# AIに続きを書かせる
output = model.generate(input_text, max_new_tokens=50, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)

## SAE Lensのロード（ブロックごとに変更）

In [ ]:
import os
import torch
from sae_lens import SAE

# 1. トークナイザーの警告を消す（おまじない）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 2. デバイス設定
device = "mps" if torch.backends.mps.is_available() else "cpu"

# 3. レンズ情報の指定
release = "gemma-scope-2b-pt-res"
sae_id = "layer_20/width_16k/average_l0_71"

print(f"🔬 {sae_id} をロード中...")

# idから場所を勝手に特定する

try:
    layer_num = sae_id.split("/")[0].split("_")[1] # "layer_20" -> "20"
    layer_name = f"blocks.{layer_num}.hook_resid_post"
    print(f"📍 自動特定: これは第 {layer_num} 層用のレンズですね。")
    print(f"   接続先: {layer_name}")
except:
    print("⚠️ 層番号の自動特定に失敗しました。手動設定が必要です。")
    layer_name = "blocks.20.hook_resid_post" # フォールバック

# ----------------------------

# 4. 【重要】3点セットを明示的に取得する新しい書き方
# from_pretrained ではなく、 _with_cfg_and_sparsity を使います
sae, cfg_dict, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
    release=release,
    sae_id=sae_id,
    device=device
)

print("\n✅ 準備完了！")
print(f"・SAE本体: ロードOK")
print(f"・設定データ: {cfg_dict['d_sae']} 個の特徴量を持っています")
print(f"・疎性データ: L0 = {sparsity}")

## SAE Lensのロードと結果（ブロックごとに変更）

In [ ]:
import torch
import os
from sae_lens import SAE
from transformer_lens import HookedTransformer

# ==========================================
# 1. 解析設定 (Configuration)
# ==========================================
# 解析対象のSAE ID
# ※ここを変更するだけで、自動的に対象の層（Layer）が切り替わります
SAE_RELEASE = "gemma-scope-2b-pt-res"
SAE_ID = "layer_20/width_16k/average_l0_71"

# 解析したいテキスト
INPUT_TEXT = "私はステーキである。"

# ==========================================
# 2. 環境セットアップ (Setup)
# ==========================================
# トークナイザーの並列化によるデッドロック防止（Mac/Linux向け安全策）
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# デバイスの自動判定 (Mac MPS / CPU)
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"🚀 使用デバイス: {device}")

# ==========================================
# 3. 接続ポイントの自動特定 (Auto-Resolution)
# ==========================================
# SAE ID文字列から「層番号」を抽出し、モデル内の正しいフック名を生成します
try:
    # "layer_20/..." -> "20" を抽出
    layer_num = SAE_ID.split("/")[0].split("_")[1]
    HOOK_POINT = f"blocks.{layer_num}.hook_resid_post"
    print(f"📍 解析対象: 第 {layer_num} 層 (Hook Point: {HOOK_POINT})")
except Exception as e:
    print(f"⚠️ 層番号の自動特定に失敗しました。デフォルト値を使用します: {e}")
    HOOK_POINT = "blocks.20.hook_resid_post"

# ==========================================
# 4. モデルとSAEのロード (Loading)
# ==========================================
print(f"🔄 SAE ({SAE_ID}) をロード中...")
sae, cfg_dict, sparsity = SAE.from_pretrained_with_cfg_and_sparsity(
    release=SAE_RELEASE,
    sae_id=SAE_ID,
    device=device
)

# Gemmaモデルのロード（既にメモリにある場合はスキップして高速化）
if 'model' not in locals():
    print("🧠 Gemmaモデルをロード中...")
    model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device)

# ==========================================
# 5. 特徴量解析の実行 (Execution)
# ==========================================
# (1) テキストをモデルに入力し、指定層の生のアクティベーション（脳波）を取得
_, cache = model.run_with_cache(INPUT_TEXT, prepend_bos=True)
original_act = cache[HOOK_POINT]

# (2) SAEを通して、アクティベーションを「意味のある特徴量」に分解
feature_acts = sae.encode(original_act)

#実際に反応した（0より大きい）特徴量の総数を数える
active_count = (feature_acts[0, -1, :] > 0).sum().item()
print(f"📊 全体のアクティブな特徴量数: {active_count} 個 / 16384 個中")
print(f"   (残りの {16384 - active_count} 個は完全に沈黙しています)")

# (3) トップｋ個を表示してみる (ｋ…上から強い順)
top_k = 70
top_values, top_indices = torch.topk(feature_acts[0, -1, :], k=top_k)

    
# ==========================================
# 6. 結果の表示 (Output)
# ==========================================
print(f"\n📖 入力テキスト: 「{INPUT_TEXT}」")
print(f"\n🔬 観察結果 (トップ {top_k}):")
print("-" * 60)

for i in range(top_k):
    feature_id = top_indices[i].item()
    strength = top_values[i].item()
    
    # 強度が弱すぎる場合はカッコ書きにする演出
    status = "🔥" if strength > 5.0 else "  "
    
    # Neuronpedia用のリンク生成（ID内のスラッシュをハイフンに置換）
    width = "16k" # 今回は16kを使っているため固定
    
    neuronpedia_id = f"{layer_num}-gemmascope-res-{width}"
    url = f"https://www.neuronpedia.org/gemma-2-2b/{neuronpedia_id}/{feature_id}"
    
    print(f"{status} Rank {i+1:<2} | ID: {feature_id:<6} | 強度: {strength:.2f}")
    print(f"   🔗解説リンク {url}")
    print("-" * 60)

    # AIに続きを書かせる
output = model.generate(input_text, max_new_tokens=100, temperature=0.7)

print("\n--- Gemmaの回答 ---")
print(output)